# CrisisLens · ResNet50 Linear Probe Training (Kaggle GPU)

訓練 ResNet50 Linear Probe 災情分類模型，產出可下載的 `.pth` + `.json`，直接放回本地 `models/` 目錄即可被 streamlit 載入。

## 執行前準備

1. **右側 Notebook options → Accelerator → GPU T4 x2**（或 P100）
2. **Internet → On**（首次需下載 torchvision 的 ImageNet 預訓練權重）
3. **Add data → 搜尋 `disaster images dataset`**：找包含 `Damaged_Infrastructure` / `Fire_Disaster` / `Land_Disaster` / `Water_Disaster` / `Non_Damage` 五個資料夾的資料集（常見的是 `varpit94/disaster-images-dataset-cnn-model`），attach 進來
4. 在下方 **Config** cell 確認 `DATA_DIR` 指向正確路徑（attach 後 Kaggle 會自動把路徑顯示在右側 Input 面板）

## 與本地 `models/train_resnet.py` 的差異

| 項目 | 本地腳本 | 本 Notebook |
|---|---|---|
| 設備 | CPU 為主 | GPU + AMP 混合精度 |
| Epochs | 5 | 10（GPU 夠快） |
| DataLoader workers | 0 | 2（pin_memory=True） |
| 評估 | classification_report | 加 confusion matrix |

**輸出檔案完全相容本地推論程式**（[models/resnet_baseline.py](models/resnet_baseline.py)），不需要修改任何程式碼。

## 1. 環境檢查

In [ ]:
import sys, torch
print(f"Python:     {sys.version.split()[0]}")
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.version.cuda}")
print(f"GPU 可用:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 名稱:   {torch.cuda.get_device_name(0)}")
    print(f"GPU 記憶體: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n⚠️ 沒偵測到 GPU — 請從右側 Notebook options 啟用 GPU 後 Restart Session")

## 2. Config

需要修改 `DATA_DIR` 的話改這格即可。其餘超參數沿用本地 [utils/config.py](utils/config.py) 的設定。

In [ ]:
from pathlib import Path

# ── 資料集路徑（依右側 Input 面板顯示的實際路徑調整）──────────
DATA_DIR = "/kaggle/input/disaster-images-dataset-cnn-model"

# ── 輸出路徑（Kaggle working dir，跑完從右側 Output 下載）─────
OUT_DIR      = Path("/kaggle/working")
WEIGHTS_PATH = OUT_DIR / "resnet50_linear.pth"
MAPPING_PATH = OUT_DIR / "resnet50_linear_classes.json"
CURVE_PATH   = OUT_DIR / "resnet_training_curve.png"
CM_PATH      = OUT_DIR / "confusion_matrix.png"

# ── 訓練超參數 ─────────────────────────────────────────────
BATCH_SIZE    = 32
LEARNING_RATE = 1e-3
EPOCHS        = 10
NUM_WORKERS   = 2
SEED          = 42
VAL_RATIO     = 0.2

# ── 資料夾名稱 → 中文類別標籤（與本地 train_resnet.py 一致）──
FOLDER_TO_ZH = {
    "Damaged_Infrastructure": "地震或建築損壞",
    "Fire_Disaster":          "火災",
    "Land_Disaster":          "土石流或坍方",
    "Water_Disaster":         "淹水",
    "Non_Damage":             "其他或無明顯災害",
}

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR:  {OUT_DIR}")

## 3. Imports

In [ ]:
import os, json, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # 允許讀取截斷/輕微損壞的圖片
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(SEED)
np.random.seed(SEED)

## 4. 確認資料集結構

Kaggle 的資料集解壓後有時候會多包一層資料夾，下面這段會自動找出含五個類別資料夾的那層；找不到會明確告訴你目前看到什麼。

In [ ]:
def find_data_root(base: str) -> str:
    base_path = Path(base)
    if not base_path.exists():
        raise FileNotFoundError(f"{base} 不存在 — 請確認資料集已 attach 並更新 DATA_DIR")
    expected = set(FOLDER_TO_ZH.keys())

    def matches(p: Path) -> bool:
        if not p.is_dir():
            return False
        subs = {x.name for x in p.iterdir() if x.is_dir()}
        return expected.issubset(subs)

    if matches(base_path):
        return str(base_path)
    for sub in base_path.rglob("*"):
        if matches(sub):
            return str(sub)
    raise FileNotFoundError(
        f"在 {base} 下找不到包含 {expected} 五個資料夾的層級。\n"
        f"請手動檢查資料集結構並修改 DATA_DIR。"
    )

DATA_ROOT = find_data_root(DATA_DIR)
print(f"✅ 資料根目錄: {DATA_ROOT}\n")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
print("類別數量：")
total_imgs = 0
for folder, zh in FOLDER_TO_ZH.items():
    fp = Path(DATA_ROOT) / folder
    if fp.exists():
        n = sum(1 for p in fp.rglob("*") if p.suffix.lower() in IMG_EXTS)
        total_imgs += n
        print(f"  {folder:25s} → {zh:12s}  {n:5d} 張")
    else:
        print(f"  {folder:25s} → ❌ 不存在")
print(f"\n總圖片數: {total_imgs}")

## 5. Transforms

Train 用 augmentation，Val 用單純 resize/crop（與本地腳本一致）。

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## 6. Dataset & DataLoader

用 `WeightedRandomSampler` 處理類別不平衡（`Non_Damage` 通常會佔最多）。

In [ ]:
full_ds     = datasets.ImageFolder(DATA_ROOT, transform=train_tf)
classes_en  = full_ds.classes
classes_zh  = [FOLDER_TO_ZH.get(c, c) for c in classes_en]
num_classes = len(classes_en)

print("類別順序（ImageFolder 自動字母排序）：")
for i, (en, zh) in enumerate(zip(classes_en, classes_zh)):
    print(f"  [{i}] {en:25s} → {zh}")

# Train/Val split
n_total = len(full_ds)
n_val   = int(n_total * VAL_RATIO)
n_train = n_total - n_val
train_ds, val_ds = torch.utils.data.random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)
val_ds.dataset.transform = val_tf   # val 不做 augmentation

def make_weighted_sampler(dataset, indices):
    targets = [dataset.targets[i] for i in indices]
    class_counts  = np.bincount(targets, minlength=num_classes)
    class_weights = 1.0 / np.maximum(class_counts.astype(float), 1.0)
    sample_weights = [class_weights[t] for t in targets]
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

sampler = make_weighted_sampler(full_ds, train_ds.indices)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"\nTrain: {n_train}  Val: {n_val}  Classes: {num_classes}")

## 7. Model

ResNet50 backbone 凍結（用 ImageNet1K V2 預訓練），只訓練 `model.fc` 一層。

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for p in model.parameters():
    p.requires_grad = False              # 凍結 backbone
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

optimizer = torch.optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)
loss_fn   = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
scaler    = torch.amp.GradScaler("cuda") if device == "cuda" else None

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"可訓練參數: {trainable:,} / {total:,}  ({100*trainable/total:.3f}%)")

## 8. 訓練

GPU + AMP 下，T4 約每個 epoch 1–3 分鐘（視資料量）。

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_acc": []}
all_pred, all_true = [], []   # 留下最後一輪的 val 預測供 confusion matrix

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Train ────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast("cuda"):
                logits = model(imgs)
                loss   = loss_fn(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = loss_fn(model(imgs), labels)
            loss.backward()
            optimizer.step()
        train_loss += loss.item()
    scheduler.step()
    train_loss /= len(train_loader)

    # ── Validate ─────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    correct = total_n = 0
    epoch_pred, epoch_true = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(imgs)
            val_loss += loss_fn(logits, labels).item()
            preds = logits.argmax(1)
            correct += (preds == labels).sum().item()
            total_n += labels.size(0)
            epoch_pred.extend(preds.cpu().tolist())
            epoch_true.extend(labels.cpu().tolist())
    val_loss /= len(val_loader)
    val_acc  = correct / total_n

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    all_pred, all_true = epoch_pred, epoch_true

    print(f"Epoch {epoch:2d}/{EPOCHS}  "
          f"Train Loss: {train_loss:.4f}  "
          f"Val Loss: {val_loss:.4f}  "
          f"Val Acc: {val_acc:.4f}  "
          f"({time.time()-t0:.1f}s)")

## 9. 驗證集評估

Per-class precision / recall / F1 + 混淆矩陣。

In [ ]:
print(classification_report(
    all_true, all_pred,
    target_names=classes_zh,
    zero_division=0,
    digits=3,
))

cm = confusion_matrix(all_true, all_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes_zh, yticklabels=classes_zh, ax=ax,
            cbar_kws={"label": "Count"})
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (Validation)")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(CM_PATH, dpi=120, bbox_inches="tight")
plt.show()

## 10. 儲存權重 + 類別對照

`resnet50_linear.pth` 與 `resnet50_linear_classes.json` **格式完全相容本地** [models/resnet_baseline.py](models/resnet_baseline.py) **的載入邏輯**，不需要修改任何程式碼。

In [ ]:
# 1. 權重檔
torch.save(model.state_dict(), WEIGHTS_PATH)
print(f"✅ 權重已存: {WEIGHTS_PATH}  ({WEIGHTS_PATH.stat().st_size / 1e6:.1f} MB)")

# 2. 類別對照（推論時要載）
mapping = {
    "classes":      classes_en,
    "zh_labels":    classes_zh,
    "class_to_idx": full_ds.class_to_idx,
}
with open(MAPPING_PATH, "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)
print(f"✅ 類別對照已存: {MAPPING_PATH}")
print(json.dumps(mapping, ensure_ascii=False, indent=2))

## 11. 訓練曲線

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="Train Loss", color="#38bdf8", linewidth=2)
ax1.plot(history["val_loss"],   label="Val Loss",   color="#f87171", linewidth=2)
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history["val_acc"], color="#4ade80", linewidth=2, marker="o")
ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylim(0, 1)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CURVE_PATH, dpi=120, bbox_inches="tight")
plt.show()
print(f"✅ 訓練曲線已存: {CURVE_PATH}")

## 12. 下載連結

點下方檔名即可下載；或從右側 **Output** 面板選 **⋮ → Download**。

**下載後放到本地專案的 [models/](models/) 目錄**（只要前兩個檔，後兩個是視覺化）：

```
CrisisLens/
└── models/
    ├── resnet50_linear.pth          ← 從這放進去
    └── resnet50_linear_classes.json ← 從這放進去
```

重啟本地 streamlit，側邊欄選 ResNet50 模式 → UI 應顯示「✅ 已訓練」。

In [ ]:
from IPython.display import FileLink, display

print("可下載檔案：\n")
for path in [WEIGHTS_PATH, MAPPING_PATH, CURVE_PATH, CM_PATH]:
    if path.exists():
        size = path.stat().st_size / 1e6
        print(f"  {path.name:35s}  {size:6.2f} MB")
        display(FileLink(str(path)))